# People Count
## In local device

## Importing required libraries

In [1]:
from ultralytics import YOLO
import cv2
import cvzone
import math
import csv
import torch
import numpy as np

## Importing Model

In [ ]:
model = YOLO("object_detection/YOLO_WEIGHTS/yolov8n.pt")  # Load a model

## Importing total classes of model

In [ ]:
total_classes = []
with open("object_detection/classes.csv", mode='r') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        #print(row[0])
        total_classes.append(row[0])

## Passing it to the model
### Only for detection

In [ ]:
cap = cv2.VideoCapture("object_detection/People_count/Dataset/people_couting_1.mp4")
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
frame_fps = int(cap.get(5))
frame_count = 0
#print(frame_width, frame_height, frame_fps)
mask = cv2.imread("object_detection/People_count/Dataset/mask_1.png")
mask = cv2.resize(mask, (640, 360))
while True:
    success, img = cap.read()
    if not success:
        break
    frame_count += 1
    #if frame_count > 2:
        #break
    img = cv2.resize(img, (640, 360))
    imgRegion = cv2.bitwise_and(img, mask)
    results = model(imgRegion, stream=True)
    #print("Results: {results}")
    for r in results:
        #print(f"r: {r}")
        boxes = r.boxes
        #print(f"boxes: {boxes}")
        for box in boxes:
            #print(f"box: {box}")
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            #print(x1, y1, x2, y2)
            conf = math.ceil((box.conf[0] * 100)) / 100
            clls = int(box.cls[0])
            current_class = total_classes[clls]
            if current_class == "person" and conf > 0.3:
                cvzone.cornerRect(img, (x1, y1, x2-x1, y2-y1))
                cvzone.putTextRect(img, f'{conf}{current_class}', (max(0, x1), max(30, y1)), scale=1, thickness=1)
            #print(f"conf: {conf}")
    cv2.imshow("Images",img)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == 27:
        break
cap.release()
cv2.destroyAllWindows()

## Passing it to model for tracking
### assign the unique id to each person

## Cloning the repo

In [4]:
# cloning repo
!git clone https://github.com/abewley/sort.git

fatal: destination path 'sort' already exists and is not an empty directory.


## Installing dependencies required for sort

In [5]:
!pip install filterpy==1.4.5
!pip install lap>=0.5.0
!pip install scikit-image>=0.19.0


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [6]:
import sys
import os
current_dir = os.getcwd()
sort_path = os.path.join(current_dir, 'sort')
if sort_path not in sys.path:
    sys.path.append(sort_path)
sys.path

['',
 '/opt/ros/jazzy/lib/python3.12/site-packages',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '/home/ritu/dev-env/lib/python3.12/site-packages',
 '/home/ritu/object_detection/People_count/sort']

In [ ]:
# Import the Sort class from the sort module
from sort import *
from sort.sort import Sort


In [8]:
trackers = Sort(max_age=20, min_hits=3, iou_threshold=0.3)

In [ ]:
cap = cv2.VideoCapture("object_detection/People_count/Dataset/people_couting_1.mp4")
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
frame_fps = int(cap.get(5))
frame_count = 0
#print(frame_width, frame_height, frame_fps)
mask = cv2.imread("object_detection/People_count/Dataset/mask_1.png")
mask = cv2.resize(mask, (640, 360))
while True:
    success, img = cap.read()
    if not success:
        break
    frame_count += 1
    #if frame_count > 2:
        #break
    detections = np.empty((0, 5))
    img = cv2.resize(img, (640, 360))
    imgRegion = cv2.bitwise_and(img, mask)
    results = model(imgRegion, stream=True)
    #print("Results: {results}")
    for r in results:
        #print(f"r: {r}")
        boxes = r.boxes
        #print(f"boxes: {boxes}")
        for box in boxes:
            #print(f"box: {box}")
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            #print(x1, y1, x2, y2)
            conf = math.ceil((box.conf[0] * 100)) / 100
            clls = int(box.cls[0])
            current_class = total_classes[clls]
            if current_class == "person" and conf > 0.3:
                #cvzone.cornerRect(img, (x1, y1, x2-x1, y2-y1))
                #cvzone.putTextRect(img, f'{conf}{current_class}', (max(0, x1), max(30, y1)), scale=1, thickness=1)
                current_array = np.array([x1, y1, x2, y2, conf])
                detections = np.vstack((detections, current_array))
    results_tracker = trackers.update(detections)
    for result in results_tracker:
        x1, y1, x2, y2, id = result
        x1, y1, x2, y2, id = int(x1), int(y1), int(x2), int(y2), int(id)
        w, h = x2 - x1, y2 - y1
        cvzone.cornerRect(img, (x1, y1, w, h), colorR=(255,0,0), colorC=(0,0,255))
        cvzone.putTextRect(img, f'{id}', (max(0, x1), max(30, y1)), scale = 2, thickness=2)
    cv2.imshow("Images",img)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == 27:
        break
cap.release()
cv2.destroyAllWindows()

## Final Counting the people

### Checking lines

In [ ]:
cap = cv2.VideoCapture("object_detection/People_count/Dataset/people_couting_1.mp4")
limits = [190, 105, 190, 360]
while True:
    success, img = cap.read()
    if not success:
        break
    img = cv2.resize(img, (640, 360))
    cv2.line(img, (limits[0], limits[1]), (limits[2], limits[3]), (0, 0, 255), 3)
    cvzone.putTextRect(img, "Line", (limits[0]+10, limits[1]-10), scale=2, thickness=2, offset=10)

    cv2.imshow("Image", img)
    key = cv2.waitKey(0) & 0xFF
    if key == ord('q'):
        break
    elif key == 27:
        break
cap.release()
cv2.destroyAllWindows()

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/ritu/dev-env/lib/python3.12/site-packages/cv2/qt/plugins"


### Creating Video Writing object

In [14]:
frame_width = 640
frame_height = 360
frame_fps = 24
out = cv2.VideoWriter("output.mp4",                          # output file name and format
                      cv2.VideoWriter_fourcc(*'mp4v'),       # video compression format
                      frame_fps, (frame_width, frame_height)) # same properties

In [15]:
print(frame_width,frame_height,frame_fps)

640 360 24


In [ ]:
cap = cv2.VideoCapture("object_detection/People_count/Dataset/people_couting_1.mp4")
limits = [190, 105, 190, 360]
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
frame_fps = int(cap.get(5))
frame_count = 0
total_count = []
#print(frame_width, frame_height, frame_fps)
mask = cv2.imread("object_detection/People_count/Dataset/mask_1.png")
mask = cv2.resize(mask, (640, 360))
while True:
    success, img = cap.read()
    if not success:
        break
    frame_count += 1
    #if frame_count > 2:
        #break
    detections = np.empty((0, 5))
    img = cv2.resize(img, (640, 360))
    imgRegion = cv2.bitwise_and(img, mask)
    results = model(imgRegion, stream=True)
    #print("Results: {results}")
    for r in results:
        #print(f"r: {r}")
        boxes = r.boxes
        #print(f"boxes: {boxes}")
        for box in boxes:
            #print(f"box: {box}")
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            #print(x1, y1, x2, y2)
            conf = math.ceil((box.conf[0] * 100)) / 100
            clls = int(box.cls[0])
            current_class = total_classes[clls]
            if current_class == "person" and conf > 0.3:
                #cvzone.cornerRect(img, (x1, y1, x2-x1, y2-y1))
                #cvzone.putTextRect(img, f'{conf}{current_class}', (max(0, x1), max(30, y1)), scale=1, thickness=1)
                current_array = np.array([x1, y1, x2, y2, conf])
                detections = np.vstack((detections, current_array))
    results_tracker = trackers.update(detections)
    cv2.line(img, (limits[0], limits[1]), (limits[2], limits[3]), (0, 0, 255), 2)
    for result in results_tracker:
        x1, y1, x2, y2, id = result
        x1, y1, x2, y2, id = int(x1), int(y1), int(x2), int(y2), int(id)
        w, h = x2 - x1, y2 - y1
        cx, cy = x1 + w // 2, y1 + h // 2
        cv2.circle(img, (cx, cy),2,(0,0,255),cv2.FILLED)
        if limits[1]<cy<limits[3] and limits[0]-5 <cx <limits[2]+5:
            if total_count.count(id)==0:
                total_count.append(id)
                cv2.line(img, (limits[0], limits[1]), (limits[2], limits[3]), (0, 255, 0), 2)
        cvzone.cornerRect(img, (x1, y1, w, h), colorR=(255,0,0), colorC=(0,0,255))
        cvzone.putTextRect(img, f'{id}', (max(0, x1), max(30, y1)), scale = 2, thickness=2)
        cvzone.putTextRect(img, f'Count: {len(total_count)}',(40,40), scale=2)
    cv2.imshow("Images",img)
    out.write(img)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == 27:
        break
cap.release()
out.release()
cv2.destroyAllWindows()